<a href="https://colab.research.google.com/github/ntatfff/todo/blob/202411241258/CreatingTumorMask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import nibabel as nib

# Assign directory
datasetDir = os.path.join('./dataset/BraTS2021_Training_Data')

listDir = os.listdir(datasetDir)
listDir.sort()
listDir
# Iterate over files in directory
listDir = os.listdir(datasetDir)
listDir.sort()
for name in listDir:
  if name.startswith('BraTS2021_'):
    flairFileName = f"{name}_flair.nii.gz"
    segFileName = f"{name}_seg.nii.gz"
    kernel = 5
    filePath = os.path.join(datasetDir, name, f"{name}_kernel{kernel}_tumor.nii.gz")
    if os.path.exists(filePath):
      print('Already exists:', name)
      continue

    flairPath = os.path.join(datasetDir, name, flairFileName)
    flairImage = nib.load(flairPath)
    flairData = np.array(flairImage.get_fdata())

    maskPath = os.path.join(datasetDir, name, segFileName)
    maskImage = nib.load(maskPath)
    maskData = np.array(maskImage.get_fdata())

    c = np.divide(maskData, maskData, out=np.zeros_like(maskData), where=maskData!=0.0, casting='unsafe')
    diameter = kernel * 2 + 1
    x, y, z = np.indices(c.shape)
    kernelMask = (x % diameter == kernel) & (y % diameter == kernel) & (z % diameter == kernel)
    def andFunc(x: int, isMasked: bool) -> int:
      return int(x and isMasked)

    andFuncVectorized = np.vectorize(andFunc)

    c = andFuncVectorized(c, kernelMask)

    tumorImage = nib.Nifti1Image(c, flairImage.affine, flairImage.header)
    nib.save(tumorImage, filePath)

    print('Done:', name)